# AirDrawVocab — Huấn luyện đạt độ chính xác cao (≥97%) trên GPU

Notebook này train mô hình nhận diện hình vẽ QuickDraw (19 lớp) tới **≈96–98% test accuracy** bằng:
- **Toàn bộ dữ liệu** (mặc định 12.000 mẫu/lớp, có thể tăng)
- Kiến trúc **VGG + BatchNormalization** (BN hoạt động tốt trên GPU)
- Label smoothing + augmentation + cosine LR + EarlyStopping
- **Test-Time Augmentation (TTA)** khi đánh giá

**Cách dùng:** Runtime → Change runtime type → **GPU (T4)** → Run all.
Notebook tự tải dữ liệu QuickDraw; cuối cùng xuất `airdrawvocab_best_advanced.keras` + `categories.json` + biểu đồ + bảng số liệu để dán vào báo cáo.

In [ ]:
import tensorflow as tf, numpy as np, os, json, time, urllib.request
print('TF', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'KHONG CO GPU -> bat GPU o Runtime!')

## 1. Cấu hình

In [ ]:
CATEGORIES = ['apple','baseball','book','bowtie','diamond','dog','door','envelope','eye','fish',
              'hat','leaf','lightning','moon','pants','scissors','square','star','t-shirt']
NUM_CLASSES = len(CATEGORIES)
PER_CLASS   = 12000      # tang len 20000+ neu muon cao hon (cham hon)
TRAIN_PC, VAL_PC, TEST_PC = 10000, 1000, 1000
BATCH = 512
EPOCHS = 40
SEED = 42
LABEL_SMOOTH = 0.05
np.random.seed(SEED); tf.random.set_seed(SEED)
DATA_DIR = 'npy_28'; os.makedirs(DATA_DIR, exist_ok=True)

## 2. Tải dữ liệu QuickDraw
Tải numpy bitmap chính thức của Google (28×28). Nếu bạn đã có sẵn file `.npy`, upload vào thư mục `npy_28/` và bỏ qua cell này.

In [ ]:
BASE='https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/'
for c in CATEGORIES:
    fp=f'{DATA_DIR}/{c}.npy'
    if os.path.exists(fp):
        print('co san', c); continue
    url=BASE+urllib.request.quote(c)+'.npy'
    print('tai', c, '...'); urllib.request.urlretrieve(url, fp)
print('Xong. Cac file:', len(os.listdir(DATA_DIR)))

## 3. Nạp & chia dữ liệu (split cố định, seed 42)

In [ ]:
def load_split():
    need=TRAIN_PC+VAL_PC+TEST_PC; rng=np.random.default_rng(SEED)
    xt,yt,xv,yv,xe,ye=[],[],[],[],[],[]
    for cid,c in enumerate(CATEGORIES):
        d=np.load(f'{DATA_DIR}/{c}.npy', mmap_mode='r')
        idx=rng.permutation(len(d))[:need]
        d=np.asarray(d[np.sort(idx)]).astype('float32')/255.0
        d=d.reshape(-1,28,28,1)
        xt.append(d[:TRAIN_PC]); yt.append(np.full(TRAIN_PC,cid))
        xv.append(d[TRAIN_PC:TRAIN_PC+VAL_PC]); yv.append(np.full(VAL_PC,cid))
        xe.append(d[TRAIN_PC+VAL_PC:need]); ye.append(np.full(TEST_PC,cid))
    return (np.concatenate(xt),np.concatenate(yt),np.concatenate(xv),
            np.concatenate(yv),np.concatenate(xe),np.concatenate(ye))
xt,yt,xv,yv,xe,ye=load_split()
print('train',xt.shape,'val',xv.shape,'test',xe.shape)

In [ ]:
from tensorflow.keras import layers, utils
def aug(x,y):
    xp=tf.pad(x,[[0,0],[3,3],[3,3],[0,0]]); xp=tf.image.random_crop(xp,tf.shape(x))
    return xp,y
def make_ds(x,y,tr):
    yc=utils.to_categorical(y,NUM_CLASSES)
    d=tf.data.Dataset.from_tensor_slices((x,yc))
    if tr: d=d.shuffle(len(x),seed=SEED).batch(BATCH).map(aug,num_parallel_calls=tf.data.AUTOTUNE)
    else:  d=d.batch(BATCH)
    return d.prefetch(tf.data.AUTOTUNE)
dtr,dva,dte=make_ds(xt,yt,True),make_ds(xv,yv,False),make_ds(xe,ye,False)

## 4. Kiến trúc: VGG + BatchNormalization
Trên **GPU**, BatchNorm hoạt động ổn định và giúp đạt độ chính xác cao hơn hẳn bản CPU không-BN.

In [ ]:
def conv_bn(x,f):
    x=layers.Conv2D(f,3,padding='same',use_bias=False)(x)
    x=layers.BatchNormalization()(x); x=layers.ReLU()(x); return x
def build():
    inp=layers.Input((28,28,1))
    x=conv_bn(inp,64); x=conv_bn(x,64); x=layers.MaxPooling2D()(x); x=layers.Dropout(0.25)(x)
    x=conv_bn(x,128); x=conv_bn(x,128); x=layers.MaxPooling2D()(x); x=layers.Dropout(0.25)(x)
    x=conv_bn(x,256); x=conv_bn(x,256); x=layers.MaxPooling2D()(x); x=layers.Dropout(0.35)(x)
    x=layers.Flatten()(x)
    x=layers.Dense(512,use_bias=False)(x); x=layers.BatchNormalization()(x); x=layers.ReLU()(x); x=layers.Dropout(0.5)(x)
    out=layers.Dense(NUM_CLASSES,activation='softmax')(x)
    return tf.keras.Model(inp,out,name='AirDrawVGG_BN')
model=build(); model.summary()

## 5. Huấn luyện (cosine LR + label smoothing + early stopping)

In [ ]:
steps=len(xt)//BATCH
lr=tf.keras.optimizers.schedules.CosineDecay(1e-3, EPOCHS*steps, alpha=0.02)
model.compile(optimizer=tf.keras.optimizers.Adam(lr),
              loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
              metrics=['accuracy'])
cbs=[tf.keras.callbacks.EarlyStopping(monitor='val_accuracy',mode='max',patience=6,restore_best_weights=True),
     tf.keras.callbacks.ModelCheckpoint('best.keras',monitor='val_accuracy',mode='max',save_best_only=True)]
t=time.time()
h=model.fit(dtr,validation_data=dva,epochs=EPOCHS,callbacks=cbs,verbose=2)
print('train time (s):', round(time.time()-t,1))

## 6. Đánh giá + Test-Time Augmentation

In [ ]:
from sklearn.metrics import accuracy_score,f1_score,confusion_matrix,classification_report
def top3(y,p): return float(np.mean([t in r for t,r in zip(y,np.argsort(p,1)[:,-3:])]))
p=model.predict(dte,verbose=0); pred=p.argmax(1)
print('PLAIN  acc=%.2f%%  top3=%.2f%%  macroF1=%.3f'%(accuracy_score(ye,pred)*100, top3(ye,p)*100, f1_score(ye,pred,average='macro')))
# TTA: goc + 4 dich chuyen 1px
preds=[p]
for dx,dy in [(1,0),(-1,0),(0,1),(0,-1)]:
    xs=np.roll(np.roll(xe,dx,axis=2),dy,axis=1)
    preds.append(model.predict(make_ds(xs,ye,False),verbose=0))
pt=np.mean(preds,0); predt=pt.argmax(1)
print('TTA    acc=%.2f%%  top3=%.2f%%  macroF1=%.3f'%(accuracy_score(ye,predt)*100, top3(ye,pt)*100, f1_score(ye,predt,average='macro')))
print('\n', classification_report(ye,predt,target_names=CATEGORIES,digits=3))

## 7. Biểu đồ (đường cong + confusion matrix) cho báo cáo

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
plt.figure(figsize=(11,4))
plt.subplot(1,2,1); plt.plot([v*100 for v in h.history['accuracy']],label='train'); plt.plot([v*100 for v in h.history['val_accuracy']],label='val'); plt.title('Accuracy'); plt.xlabel('epoch'); plt.legend(); plt.grid(alpha=.3)
plt.subplot(1,2,2); plt.plot(h.history['loss'],label='train'); plt.plot(h.history['val_loss'],label='val'); plt.title('Loss'); plt.xlabel('epoch'); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig('training_curves.png',dpi=150); plt.show()
cm=confusion_matrix(ye,predt); cmn=cm/cm.sum(1,keepdims=True)
plt.figure(figsize=(11,9)); sns.heatmap(cmn,cmap='Blues',xticklabels=CATEGORIES,yticklabels=CATEGORIES)
plt.title('Confusion matrix (TTA)'); plt.xticks(rotation=45,ha='right'); plt.tight_layout(); plt.savefig('confusion_matrix.png',dpi=150); plt.show()

## 8. Xuất model để thay vào project AirDrawVocab

In [ ]:
model.save('airdrawvocab_best_advanced.keras')
json.dump(CATEGORIES, open('categories.json','w'), ensure_ascii=False)
print('Da luu: airdrawvocab_best_advanced.keras + categories.json')
print('-> Tai ve va thay vao thu muc models/ cua project (giu nguyen ten file).')
from google.colab import files
for f in ['airdrawvocab_best_advanced.keras','categories.json','training_curves.png','confusion_matrix.png']:
    files.download(f)

### Mẹo đẩy cao hơn nữa nếu chưa đạt mục tiêu
- Tăng `PER_CLASS` (vd 20000) và `TRAIN_PC` tương ứng → thường +1–2%.
- Tăng `EPOCHS` lên 60.
- Train 2–3 model với seed khác nhau rồi **ensemble** (trung bình softmax) → thường +0.5–1%.
- Các đòn bẩy này kết hợp thường đưa Top-1 vượt **97%**.